<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/18_GES_Aware_Genomic_RAG_Cell_7C11_Protocol_Amendment_A004_Fully_Automated_Structured_Evaluation_Freeze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, operator declaration, exact Cell 7C10 lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re
import tempfile

import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '18_GES_Aware_Genomic_RAG_Cell_7C11_'
    'Protocol_Amendment_A004_Fully_Automated_Structured_Evaluation_Freeze.ipynb'
)
CELL_ID = '7C11'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_RUNS_PER_QUESTION_CONDITION = 3
EXPECTED_CONDITIONS = 6
BOOTSTRAP_REPLICATES = 2_000

# Explicit operator declaration based on the study owner's statement.
# This cannot be inferred cryptographically from empty templates.
OPERATOR_DECLARATION_NO_HUMAN_SCORING_PERFORMED = True

if OPERATOR_DECLARATION_NO_HUMAN_SCORING_PERFORMED is not True:
    raise RuntimeError(
        'A004 may not be frozen after human scoring has begun. '
        'Stop and document the actual scoring history before proceeding.'
    )

EXPECTED_CELL_7C10_TERMINAL_DECISION = (
    'PASS_STAGE7C10_A003_HYBRID_SINGLE_REVIEWER_PACKET_MATERIALIZED_'
    'FIRSTPASS1440_RUBRIC11520_ATOMICCLAIM1440_DETERMINISTIC_INPUT1440_'
    'REPEAT160_REBLINDED_TWO_PER_QUESTION_REPEAT_RUBRIC1280_14DAY_RELEASE_GATE_'
    'LOCKED_CHECKSUM_PROTECTED_FIRSTPASS_HUMAN_REVIEW_MAY_BEGIN_NO_CONDITION_'
    'UNBLINDING_RUN_AGGREGATION_PRIMARY_ENDPOINT_BOOTSTRAP_OR_ARM_COMPARISON_'
    'NEXT_AUTOMATED_EXECUTION_NOT_AUTHORIZED'
)

CELL_7C10_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)
CELL_7C10_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)
CELL_7C10_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)

CELL_7C10 = OrderedDict([
    ('firstpass_review_packet', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_firstpass_single_reviewer_packet_v1.parquet',
        'sha256': 'e1c323cc491c133ea367ee47934ea32f95551053fa3501cc7c5169022d1c0b14',
    }),
    ('firstpass_assignment', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_firstpass_single_reviewer_assignment_v1.csv',
        'sha256': '635f2d7fc5b87e5c67a4741b63bd304d77db904a19dbe08e87a1fd09bbf148c0',
    }),
    ('firstpass_rubric_template', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_firstpass_rubric_scoring_template_v1.csv',
        'sha256': '4eef9e182f09d39296ecc584ae4c871817c101fdb02b202d2be73f897fd97b42',
    }),
    ('firstpass_atomic_claim_template', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_firstpass_atomic_claim_annotation_template_v1.csv',
        'sha256': 'd296000efa407bf3e2672ab31523d43db1bb77010983ec7cbf977f8e0380c5c3',
    }),
    ('deterministic_scoring_input', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_deterministic_scoring_input_v1.parquet',
        'sha256': '7b987c894bf82e4607ba49f270b14d68d9d4ba201e86234a02e225fd279ceb5f',
    }),
    ('repeat_sample_inventory', {
        'path': CELL_7C10_CONFIG_DIR / 'cell_7c10_repeat_sample_inventory_internal_v1.csv',
        'sha256': '8f505b6662c308eb2022c17347ddba1757b0086c0d4141b3d84e97e28f13047a',
    }),
    ('repeat_review_packet', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_repeat_reblinded_review_packet_LOCKED_v1.parquet',
        'sha256': '57d303275421ef5ddcb419d3b0880be99a0bb428328e43fbc37e8a6f35203f7f',
    }),
    ('repeat_rubric_template', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_repeat_rubric_scoring_template_LOCKED_v1.csv',
        'sha256': '0e8a1841b1222adcc309ff34f1789e8bc7cb9f135ee308c93d53cc9acac8702b',
    }),
    ('repeat_atomic_claim_template', {
        'path': CELL_7C10_EXEC_DIR / 'cell_7c10_repeat_atomic_claim_annotation_template_LOCKED_v1.csv',
        'sha256': 'd2836591874320f8b5ee46d614fb5252852368861ab77c02195576fc09fd25fe',
    }),
    ('repeat_internal_routing_map', {
        'path': CELL_7C10_CONFIG_DIR / 'cell_7c10_repeat_internal_routing_map_v1.parquet',
        'sha256': '4ab713f9eceb3bcc6e03c0de4b1f8884062f8049442dd649a4907d8228270ef1',
    }),
    ('repeat_release_gate', {
        'path': CELL_7C10_CONFIG_DIR / 'cell_7c10_repeat_release_gate_v1.json',
        'sha256': '588052a5df12cbd0ed90ed4550cc9214c7deb9a862ba89eac5739c24334e5fc9',
    }),
    ('reviewer_instructions', {
        'path': CELL_7C10_CONFIG_DIR / 'cell_7c10_single_reviewer_instructions_v1.json',
        'sha256': '7b19f783bb05335d1285e83ec043fda60bbcebf99e807145b4e74e096e24a480',
    }),
    ('input_inventory', {
        'path': CELL_7C10_CONFIG_DIR / 'cell_7c10_verified_input_inventory_v1.csv',
        'sha256': '5bed42070650938cc609114a8b08977d779c8d76f1e0510011b418655536f733',
    }),
    ('execution_report', {
        'path': CELL_7C10_QC_DIR / 'cell_7c10_packet_materialization_execution_report_v1.json',
        'sha256': 'e9b7b38f92bda45f16280a107f0d12c16125465600947f8a27c19b2b0c981857',
    }),
    ('qc', {
        'path': CELL_7C10_QC_DIR / 'cell_7c10_packet_materialization_qc_v1.json',
        'sha256': '6afeea824926b48960681201941c9918a1bcaef60250cb3632651ed3627103b4',
    }),
    ('manifest', {
        'path': CELL_7C10_CONFIG_DIR / 'cell_7c10_hybrid_single_reviewer_packet_manifest_v1.json',
        'sha256': '489bd4832f99efe5db721759d8fb63afc1eee06ef5c1e8f43e37538e6c651102',
    }),
])

A004_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)

OUTPUTS = OrderedDict([
    ('amendment',
     A004_DIR / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1.json'),
    ('endpoint_spec',
     A004_DIR / 'protocol_amendment_A004_automated_endpoint_spec_v1.json'),
    ('aggregation_inference_spec',
     A004_DIR / 'protocol_amendment_A004_aggregation_and_inference_spec_v1.json'),
    ('input_inventory',
     A004_DIR / 'protocol_amendment_A004_verified_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'protocol_amendment_A004_qc_v1.json'),
    ('manifest',
     A004_DIR / 'protocol_amendment_A004_manifest_v1.json'),
])

for directory in (A004_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C11 / A004 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'A004 directory: {A004_DIR}')
print(f'QC directory  : {QC_DIR}')
print('Operator declaration — human scoring performed before A004: NO')

A004 directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/protocol_amendment_A004_fully_automated_structured_evaluation_v1
QC directory  : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/protocol_amendment_A004_fully_automated_structured_evaluation_v1
Operator declaration — human scoring performed before A004: NO


## 2. SHA-256, sidecar, metadata, and stable-write helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


with tempfile.TemporaryDirectory(prefix='cell_7c11_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify Cell 7C10 and inspect deterministic-input schema only

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C10.items():
    record = verify_exact_artifact(
        f'cell_7c10_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C10'
    verified_inputs.append(record)

manifest_7c10 = load_json(CELL_7C10['manifest']['path'])
qc_7c10 = load_json(CELL_7C10['qc']['path'])
repeat_gate_7c10 = load_json(CELL_7C10['repeat_release_gate']['path'])

if manifest_7c10.get('terminal_decision') != EXPECTED_CELL_7C10_TERMINAL_DECISION:
    raise AssertionError('Cell 7C10 terminal PASS mismatch.')
if manifest_7c10.get('next_authorized_cell') is not None:
    raise AssertionError('Cell 7C10 unexpectedly authorized a downstream cell.')
if manifest_7c10.get('condition_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C10 unexpectedly authorized condition unblinding.')
if manifest_7c10.get('run_aggregation_authorized') is not False:
    raise AssertionError('Cell 7C10 unexpectedly authorized run aggregation.')
if manifest_7c10.get('primary_endpoint_calculation_authorized') is not False:
    raise AssertionError('Cell 7C10 unexpectedly authorized primary endpoint calculation.')
if int(qc_7c10.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C10 QC does not report zero failures.')
if repeat_gate_7c10.get('current_release_authorized') is not False:
    raise AssertionError('Cell 7C10 repeat review was unexpectedly released.')

det_meta = parquet_metadata(CELL_7C10['deterministic_scoring_input']['path'])

EXPECTED_DETERMINISTIC_COLUMNS = {
    'review_item_id',
    'question_id',
    'response_policy',
    'conflict_detected',
    'evidence_strength',
    'confidence',
    'cited_evidence_ids_json',
    'context_packet_ids_json',
    'expected_abstention_or_qualification_required',
    'deterministic_scores_calculated',
}

missing = sorted(EXPECTED_DETERMINISTIC_COLUMNS - set(det_meta['schema_names']))
if missing:
    raise AssertionError(
        'Cell 7C10 deterministic input is missing required fields: '
        + ', '.join(missing)
    )
if det_meta['rows'] != EXPECTED_RESPONSES:
    raise AssertionError(
        f'Expected 1,440 deterministic-input rows; observed {det_meta["rows"]}.'
    )

print('Cell 7C10 package                      : 16/16 exact hashes + sidecars')
print('Cell 7C10 terminal PASS                : VERIFIED')
print(f'Deterministic input rows               : {det_meta["rows"]:,}')
print('Deterministic input schema             : VERIFIED')
print('Response-level scoring performed       : NO')
print('Condition identity opened              : NO')
print('Run aggregation performed              : NO')

Cell 7C10 package                      : 16/16 exact hashes + sidecars
Cell 7C10 terminal PASS                : VERIFIED
Deterministic input rows               : 1,440
Deterministic input schema             : VERIFIED
Response-level scoring performed       : NO
Condition identity opened              : NO
Run aggregation performed              : NO


## 4. Freeze A004 automated endpoint specification

In [5]:
PRIMARY_ENDPOINT_NAME = 'automated_evidence_fidelity_pass'

endpoint_spec = {
    'protocol_amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'evaluation_mode': 'fully_automated_structured_evaluation',
    'population': {
        'frozen_responses': EXPECTED_RESPONSES,
        'primary_questions': EXPECTED_QUESTIONS,
        'conditions': EXPECTED_CONDITIONS,
        'runs_per_question_condition': EXPECTED_RUNS_PER_QUESTION_CONDITION,
    },
    'superseded_primary_endpoint': {
        'definition':
            'fraction of atomic factual claims that are both correct and citation-supported',
        'reason_superseded':
            'No human semantic review is operationally possible; automated evaluation cannot validly '
            'substitute for claim-level human factual correctness/citation-entailment judgments.',
        'human_claim_level_endpoint_will_be_reported': False,
    },
    'primary_automated_endpoint': {
        'name': PRIMARY_ENDPOINT_NAME,
        'unit': 'response',
        'type': 'binary',
        'definition':
            'Pass only when citation_integrity_pass, citation_presence_pass, conflict_concordance_pass, '
            'and caution_policy_concordance_pass are all true.',
        'components': OrderedDict([
            ('citation_integrity_pass', {
                'definition':
                    'All cited evidence IDs are members of the exact frozen five-packet context for that response. '
                    'Zero invalid/hallucinated evidence IDs are allowed.',
                'empty_evidence_list_behavior':
                    'Does not fail citation integrity by itself; citation presence is evaluated separately.',
            }),
            ('citation_presence_pass', {
                'definition':
                    'For response_policy in {answer, cautious_answer}, at least one valid context evidence ID '
                    'must be cited. For response_policy=abstain, no minimum evidence-ID count is required.',
            }),
            ('conflict_concordance_pass', {
                'definition':
                    'response conflict_detected must exactly equal the frozen answer-key-derived '
                    'aggregate conflict requirement represented in the deterministic input.',
            }),
            ('caution_policy_concordance_pass', {
                'definition':
                    'If expected_abstention_or_qualification_required=true, response_policy must be '
                    'cautious_answer or abstain. If false, answer or cautious_answer passes and abstain fails '
                    'as over-abstention.',
            }),
        ]),
        'mathematical_rule':
            'int(citation_integrity_pass AND citation_presence_pass AND '
            'conflict_concordance_pass AND caution_policy_concordance_pass)',
    },
    'secondary_automated_endpoints': [
        {
            'name': 'citation_integrity_pass',
            'type': 'binary',
        },
        {
            'name': 'citation_presence_pass',
            'type': 'binary',
        },
        {
            'name': 'invalid_or_hallucinated_evidence_id_count',
            'type': 'count',
        },
        {
            'name': 'valid_context_evidence_id_count',
            'type': 'count',
        },
        {
            'name': 'context_citation_coverage',
            'type': 'continuous_0_to_1',
            'definition':
                'distinct valid cited context packet IDs divided by five',
        },
        {
            'name': 'conflict_concordance_pass',
            'type': 'binary',
        },
        {
            'name': 'caution_policy_concordance_pass',
            'type': 'binary',
        },
        {
            'name': 'required_caution_compliance',
            'type': 'binary_or_not_applicable',
            'definition':
                'Among responses whose frozen key requires caution, whether policy is cautious_answer or abstain.',
        },
        {
            'name': 'over_abstention',
            'type': 'binary_or_not_applicable',
            'definition':
                'Among responses whose frozen key does not require caution, whether policy=abstain.',
        },
        {
            'name': 'response_policy',
            'type': 'categorical_descriptive',
        },
        {
            'name': 'evidence_strength',
            'type': 'categorical_descriptive',
        },
        {
            'name': 'confidence',
            'type': 'continuous_descriptive',
        },
    ],
    'explicitly_out_of_scope': [
        'free-text factual correctness',
        'semantic citation entailment',
        'human atomic-claim correctness',
        'clinical appropriateness',
        'patient-level safety',
        'clinical decision utility',
    ],
    'missingness_and_parse_policy': {
        'invalid_json_in_frozen_deterministic_input':
            'fail closed; Cell 7C12 stops rather than imputing',
        'missing_required_field':
            'fail closed; Cell 7C12 stops rather than imputing',
        'duplicate_cited_evidence_ids':
            'deduplicate for coverage counts but preserve a duplicate-count QC field; duplicates do not create extra credit',
    },
    'condition_identity_required_for_response_level_scoring': False,
    'score_bearing_GES_artifacts_required_for_response_level_scoring': False,
}

stable_write_json(OUTPUTS['endpoint_spec'], endpoint_spec)
write_sidecar(OUTPUTS['endpoint_spec'])

print('A004 primary endpoint                  : automated_evidence_fidelity_pass')
print('Primary endpoint unit                  : response-level binary')
print('Primary components                     : 4')
print('Human factual correctness endpoint      : SUPERSEDED — NOT REPORTED')
print('Free-text semantic correctness          : OUT OF SCOPE')

A004 primary endpoint                  : automated_evidence_fidelity_pass
Primary endpoint unit                  : response-level binary
Primary components                     : 4
Human factual correctness endpoint      : SUPERSEDED — NOT REPORTED
Free-text semantic correctness          : OUT OF SCOPE


## 5. Freeze later aggregation, unblinding, and paired inference design

In [6]:
aggregation_inference_spec = {
    'protocol_amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'sequence': [
        'Cell 7C12 computes and freezes 1,440 response-level automated outcomes while condition identity remains hidden.',
        'A separate authorization cell re-verifies the frozen automated outcomes before opening the internal routing map.',
        'Only then are blinded aliases/condition identities joined back to frozen outcomes.',
        'Three run-level response outcomes are averaged within each question-condition to produce one question-level condition score.',
        'Question-level condition scores are used for paired comparisons and bootstrap inference.',
    ],
    'run_aggregation': {
        'response_level_primary': PRIMARY_ENDPOINT_NAME,
        'within_question_condition':
            'mean of the three binary automated_evidence_fidelity_pass values across run_id 0,1,2',
        'question_level_score_range': [0.0, 1.0],
        'expected_question_condition_units': EXPECTED_QUESTIONS * EXPECTED_CONDITIONS,
        'expected_runs_per_question_condition': EXPECTED_RUNS_PER_QUESTION_CONDITION,
    },
    'primary_comparison': {
        'experimental_condition': 'D Full-GES',
        'reference_condition': 'A semantic-only',
        'estimand':
            'mean paired question-level difference: Full-GES minus semantic-only '
            'in three-run automated evidence fidelity pass proportion',
        'direction':
            'positive values favor Full-GES',
    },
    'mandatory_secondary_comparisons': [
        'D Full-GES vs B review/conflict-aware',
        'D Full-GES vs C combined-metadata',
        'D Full-GES vs E no-star-GES',
        'D Full-GES vs F random-quality',
    ],
    'bootstrap': {
        'unit': 'question',
        'paired': True,
        'replicates': BOOTSTRAP_REPLICATES,
        'sampling':
            'sample the 80 question IDs with replacement; use the same sampled question indices for both conditions in each comparison',
        'interval':
            'two-sided 95% percentile interval',
        'point_estimate':
            'mean paired question-level difference using all 80 questions',
        'no_response_level_independence_assumption_for_bootstrap': True,
    },
    'secondary_metric_inference': {
        'citation_integrity_pass': 'paired question-level bootstrap',
        'citation_presence_pass': 'paired question-level bootstrap',
        'conflict_concordance_pass': 'paired question-level bootstrap',
        'caution_policy_concordance_pass': 'paired question-level bootstrap',
        'context_citation_coverage': 'paired question-level bootstrap',
        'invalid_or_hallucinated_evidence_id_count': 'paired question-level bootstrap on question-level mean counts',
        'confidence': 'descriptive unless a later prespecified calibration target is justified',
    },
    'gene_reporting': {
        'primary_genes': ['BRCA1', 'BRCA2', 'MLH1'],
        'exploratory_gene': 'EGFR',
        'egfr_must_be_reported_separately': True,
    },
    'multiplicity': {
        'primary_D_vs_A': 'single primary comparison; no multiplicity correction',
        'D_vs_B_C_E_F':
            'mandatory secondary family; report raw paired bootstrap intervals and apply Holm correction '
            'to the four two-sided paired comparison p-values/sign-probability analogues if such p-values are computed',
    },
    'condition_unblinding_allowed_in_cell_7c11': False,
    'condition_unblinding_allowed_in_cell_7c12': False,
    'run_aggregation_allowed_in_cell_7c12': False,
    'bootstrap_allowed_in_cell_7c12': False,
}

stable_write_json(OUTPUTS['aggregation_inference_spec'], aggregation_inference_spec)
write_sidecar(OUTPUTS['aggregation_inference_spec'])

print('Future primary comparison              : D Full-GES vs A semantic-only')
print('Mandatory secondary comparisons         : D vs B/C/E/F')
print('Bootstrap unit                         : paired question level')
print('Bootstrap replicates                   : 2,000')
print('EGFR                                    : exploratory; separate')
print('Unblinding in Cell 7C11/7C12            : PROHIBITED')

Future primary comparison              : D Full-GES vs A semantic-only
Mandatory secondary comparisons         : D vs B/C/E/F
Bootstrap unit                         : paired question level
Bootstrap replicates                   : 2,000
EGFR                                    : exploratory; separate
Unblinding in Cell 7C11/7C12            : PROHIBITED


## 6. Freeze Amendment A004 and authorize blinded Cell 7C12 scoring only

In [7]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C12_FULLY_AUTOMATED_BLINDED_STRUCTURED_EVALUATION_'
    'OF_1440_FROZEN_RESPONSES_USING_A004_PRIMARY_AUTOMATED_EVIDENCE_FIDELITY_'
    'COMPOSITE_AND_PRESPECIFIED_SECONDARY_ENDPOINTS_FROM_CELL7C10_DETERMINISTIC_'
    'INPUT_ONLY_NO_HUMAN_REVIEW_CONDITION_UNBLINDING_INTERNAL_ROUTING_MAP_ACCESS_'
    'RUN_AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

amendment_payload = {
    'protocol_amendment_id': AMENDMENT_ID,
    'cell_id': CELL_ID,
    'stage': STAGE,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'amendment_status':
        'FROZEN_BEFORE_ANY_AUTOMATED_RESPONSE_LEVEL_OUTCOME_OR_CONDITION_LEVEL_PERFORMANCE_CALCULATION',
    'operator_declaration': {
        'human_scoring_performed_before_A004': False,
        'basis':
            'Explicit study-owner declaration; absence of manual scoring cannot be proven solely from frozen empty templates.',
    },
    'reason_for_amendment':
        'No human review is operationally possible. The study will not simulate or misrepresent human adjudication. '
        'A004 supersedes A003 for execution and replaces the human semantic primary endpoint with a narrower '
        'fully automated structured evidence-fidelity endpoint before any response-level automated outcome is calculated.',
    'supersedes_for_execution': {
        'protocol_amendment_A003': True,
        'cell_7c9_and_7c10_artifacts_preserved_for_audit': True,
        'repeat_assessment_path_abandoned': True,
        'human_review_files_should_not_be_scored': True,
    },
    'unchanged_upstream_experiment': {
        'questions': 80,
        'conditions': 6,
        'runs_per_question_condition': 3,
        'frozen_llm_responses': 1440,
        'retrieval_changed': False,
        'reranking_changed': False,
        'top5_context_changed': False,
        'prompts_changed': False,
        'llm_model_changed': False,
        'generation_parameters_changed': False,
        'frozen_response_content_changed': False,
    },
    'scientific_change': {
        'primary_endpoint_changed': True,
        'old_primary_endpoint':
            'fraction of atomic factual claims that are both correct and citation-supported',
        'new_primary_endpoint': PRIMARY_ENDPOINT_NAME,
        'claim_scope_narrowed': True,
        'free_text_factual_correctness_will_not_be_claimed': True,
    },
    'endpoint_spec_sha256': sha256_file(OUTPUTS['endpoint_spec']),
    'aggregation_inference_spec_sha256': sha256_file(OUTPUTS['aggregation_inference_spec']),
    'authorization_decision': authorization_decision,
    'next_authorized_cell': '7C12',
    'cell_7c12_scope':
        'Compute and freeze response-level automated structured outcomes for all 1,440 responses '
        'using only the Cell 7C10 deterministic-scoring input and A004 endpoint specification.',
    'human_review_authorized': False,
    'condition_unblinding_authorized': False,
    'internal_routing_map_access_authorized_in_7c12': False,
    'run_aggregation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
    'llm_call_authorized': False,
}

stable_write_json(OUTPUTS['amendment'], amendment_payload)
write_sidecar(OUTPUTS['amendment'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

prewrite_checks = OrderedDict([
    ('cell7c10_all_16_artifacts_verified', len(verified_inputs) == 16),
    ('cell7c10_terminal_pass_exact',
     manifest_7c10.get('terminal_decision') == EXPECTED_CELL_7C10_TERMINAL_DECISION),
    ('deterministic_input_1440', det_meta['rows'] == 1440),
    ('deterministic_schema_complete',
     EXPECTED_DETERMINISTIC_COLUMNS.issubset(set(det_meta['schema_names']))),
    ('operator_declares_no_human_scoring',
     OPERATOR_DECLARATION_NO_HUMAN_SCORING_PERFORMED is True),
    ('a003_superseded',
     amendment_payload['supersedes_for_execution']['protocol_amendment_A003'] is True),
    ('primary_endpoint_changed_transparently',
     amendment_payload['scientific_change']['primary_endpoint_changed'] is True),
    ('new_primary_endpoint_exact',
     amendment_payload['scientific_change']['new_primary_endpoint'] == PRIMARY_ENDPOINT_NAME),
    ('four_primary_components',
     len(endpoint_spec['primary_automated_endpoint']['components']) == 4),
    ('free_text_correctness_out_of_scope',
     'free-text factual correctness' in endpoint_spec['explicitly_out_of_scope']),
    ('primary_D_vs_A_frozen',
     aggregation_inference_spec['primary_comparison']['experimental_condition'] == 'D Full-GES'
     and aggregation_inference_spec['primary_comparison']['reference_condition'] == 'A semantic-only'),
    ('secondary_D_vs_B_C_E_F_count_4',
     len(aggregation_inference_spec['mandatory_secondary_comparisons']) == 4),
    ('bootstrap_2000',
     aggregation_inference_spec['bootstrap']['replicates'] == 2000),
    ('bootstrap_question_level',
     aggregation_inference_spec['bootstrap']['unit'] == 'question'),
    ('egfr_exploratory',
     aggregation_inference_spec['gene_reporting']['exploratory_gene'] == 'EGFR'),
    ('human_review_false', amendment_payload['human_review_authorized'] is False),
    ('condition_unblinding_false', amendment_payload['condition_unblinding_authorized'] is False),
    ('routing_access_7c12_false',
     amendment_payload['internal_routing_map_access_authorized_in_7c12'] is False),
    ('run_aggregation_false', amendment_payload['run_aggregation_authorized'] is False),
    ('bootstrap_false', amendment_payload['bootstrap_inference_authorized'] is False),
    ('arm_comparison_false', amendment_payload['arm_comparison_authorized'] is False),
    ('llm_call_false', amendment_payload['llm_call_authorized'] is False),
    ('no_scoring_in_cell7c11', True),
    ('no_unblinding_in_cell7c11', True),
])

failed = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C11 / A004 prewrite QC failed:\\n- ' + '\\n- '.join(failed)
    )

terminal_decision = (
    'PASS_STAGE7C11_PROTOCOL_AMENDMENT_A004_FULLY_AUTOMATED_STRUCTURED_EVALUATION_'
    'FROZEN_BEFORE_SCORING_A003_HUMAN_PATH_SUPERSEDED_PRIMARY_ENDPOINT_CHANGED_TO_'
    'AUTOMATED_EVIDENCE_FIDELITY_COMPOSITE_FOUR_COMPONENTS_DVSA_PRIMARY_DVSBCEF_'
    'SECONDARY_2000_PAIRED_QUESTION_BOOTSTRAP_LATER_CELL7C12_BLINDED_RESPONSE_LEVEL_'
    'SCORING_ONLY_AUTHORIZED_NO_HUMAN_REVIEW_CONDITION_UNBLINDING_ROUTING_ACCESS_'
    'RUN_AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c10_manifest_sha256': CELL_7C10['manifest']['sha256'],
        'cell_7c10_deterministic_scoring_input_sha256':
            CELL_7C10['deterministic_scoring_input']['sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C12',
    'human_review_authorized': False,
    'condition_unblinding_authorized': False,
    'internal_routing_map_access_authorized_in_7c12': False,
    'run_aggregation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
}

stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C11 final readback failed: {path}')

amend_rb = load_json(OUTPUTS['amendment'])
endpoint_rb = load_json(OUTPUTS['endpoint_spec'])
agg_rb = load_json(OUTPUTS['aggregation_inference_spec'])
qc_rb = load_json(OUTPUTS['qc'])
manifest_rb = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('A004_exact', amend_rb['protocol_amendment_id'] == 'A004'),
    ('new_primary_exact',
     endpoint_rb['primary_automated_endpoint']['name'] == PRIMARY_ENDPOINT_NAME),
    ('new_primary_four_components',
     len(endpoint_rb['primary_automated_endpoint']['components']) == 4),
    ('primary_D_vs_A',
     agg_rb['primary_comparison']['experimental_condition'] == 'D Full-GES'
     and agg_rb['primary_comparison']['reference_condition'] == 'A semantic-only'),
    ('bootstrap_2000', agg_rb['bootstrap']['replicates'] == 2000),
    ('next_cell_7c12', manifest_rb.get('next_authorized_cell') == '7C12'),
    ('human_review_false', manifest_rb.get('human_review_authorized') is False),
    ('unblinding_false', manifest_rb.get('condition_unblinding_authorized') is False),
    ('routing_access_false',
     manifest_rb.get('internal_routing_map_access_authorized_in_7c12') is False),
    ('run_aggregation_false', manifest_rb.get('run_aggregation_authorized') is False),
    ('bootstrap_false', manifest_rb.get('bootstrap_inference_authorized') is False),
    ('arm_comparison_false', manifest_rb.get('arm_comparison_authorized') is False),
    ('qc_zero_failures', int(qc_rb.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C11 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C11')
print('PROTOCOL AMENDMENT A004 — FULLY AUTOMATED STRUCTURED EVALUATION DESIGN FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM CELL 7C10 REVERIFICATION')
print(f'Cell 7C10 manifest SHA-256                    : {CELL_7C10["manifest"]["sha256"]}')
print('Cell 7C10 terminal PASS verified              : YES')
print('Cell 7C10 frozen artifacts                    : 16/16 exact hashes + sidecars')
print('Deterministic-scoring input                   : 1,440 rows — schema verified')
print('Human scoring performed before A004           : NO — operator declaration')

print('\\nPROTOCOL AMENDMENT A004')
print('A003 human-review execution path              : SUPERSEDED')
print('Human review                                  : NOT REQUIRED / NOT AUTHORIZED')
print('Old atomic-claim human primary endpoint       : SUPERSEDED')
print('New primary endpoint                          : automated_evidence_fidelity_pass')
print('Free-text factual correctness claim           : NOT PERMITTED')

print('\\nNEW PRIMARY AUTOMATED ENDPOINT')
print('Component 1                                   : citation integrity')
print('Component 2                                   : citation presence when answering')
print('Component 3                                   : conflict concordance')
print('Component 4                                   : caution-policy concordance')
print('Response passes primary endpoint              : ALL FOUR COMPONENTS MUST PASS')

print('\\nLATER CONDITION COMPARISON — FROZEN NOW')
print('Primary comparison                            : D Full-GES vs A semantic-only')
print('Mandatory secondary comparisons               : D vs B / C / E / F')
print('Run aggregation                               : mean of 3 response-level passes per question-condition')
print('Bootstrap                                     : 2,000 paired question-level replicates')
print('EGFR                                          : exploratory; report separately')

print('\\nCELL 7C12 AUTHORIZATION')
print('Automated response-level scoring              : AUTHORIZED — all 1,440')
print('Allowed scoring input                         : Cell 7C10 deterministic-scoring input only')
print('Condition identity unblinding                 : PROHIBITED')
print('Internal routing-map access                   : PROHIBITED')
print('Run aggregation                               : PROHIBITED')
print('Bootstrap / arm comparison                    : PROHIBITED')

print('\\nCELL 7C11 / A004 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C12                          : blinded automated response-level scoring + freeze only')
print('Human review                                  : ABANDONED')
print('Condition unblinding / comparative analysis   : NOT YET AUTHORIZED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C11
PROTOCOL AMENDMENT A004 — FULLY AUTOMATED STRUCTURED EVALUATION DESIGN FREEZE
Notebook                                      : 18_GES_Aware_Genomic_RAG_Cell_7C11_Protocol_Amendment_A004_Fully_Automated_Structured_Evaluation_Freeze.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM CELL 7C10 REVERIFICATION
Cell 7C10 manifest SHA-256                    : 489bd4832f99efe5db721759d8fb63afc1eee06ef5c1e8f43e37538e6c651102
Cell 7C10 terminal PASS verified              : YES
Cell 7C10 frozen artifacts                    : 16/16 exact hashes + sidecars
Deterministic-scoring input                   : 1,440 rows — schema verified
Human scoring performed before A004           : NO — operator declaration
\nPROTOCOL AMENDMENT A004
A003 human-review